# Berkeley Humanoid PPO Training and Commanded Locomotion

This notebook trains one shared PPO policy for the
`BerkeleyHumanoidJoystickFlatTerrain` environment.

Use **Runtime → Run all** after you select a GPU runtime.

The default smoke mode tests the complete pipeline with a short training run.
It does not produce a high-quality walking policy. Set `RUN_FULL_TRAINING` to
`True` for the tuned 150-million-step experiment.

## What this notebook does

1. Install one compatible dependency set.
2. Load the Berkeley Humanoid environment and its PPO configuration.
3. Train one actor and one critic from parallel simulator copies.
4. Save the policy parameters and learning curve.
5. run a keyboard-style command sequence and save a rollout video.

The Colab runtime is temporary. Notebook code belongs in GitHub. Trained
parameters, plots, and videos belong in Google Drive or a downloaded archive.

In [ ]:
#@title Run settings
RUN_FULL_TRAINING = False  #@param {type:"boolean"}
SAVE_ARTIFACTS_TO_DRIVE = False  #@param {type:"boolean"}
SEED = 0  #@param {type:"integer"}

# Each character becomes a velocity command for SECONDS_PER_KEY seconds.
# W/S: forward/back, A/D: left/right, Q/E: turn left/right, X: stop.
KEY_SEQUENCE = "wwwwaaddqqeexx"  #@param {type:"string"}
SECONDS_PER_KEY = 0.5  #@param {type:"number"}

In [ ]:
#@title Install one compatible package set
# Pin Brax because its 0.14.2 release uses APIs removed from JAX 0.11.
%pip install -q --upgrade "playground==0.2.0" "flax==0.12.8" "mediapy>=1.2.4" "brax @ https://github.com/google/brax/archive/c8605294264a7b7546268adbb4beabc9eb624000.tar.gz"

In [ ]:
#@title Imports and runtime verification
import functools
import gc
import importlib.metadata as metadata
import json
import os
import pickle
import subprocess
import urllib.request
from datetime import datetime, timezone
from pathlib import Path

# Set EGL before importing MuJoCo. Colab has no desktop OpenGL window.
os.environ["MUJOCO_GL"] = "egl"
os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"] = "false"

import jax
from jax import numpy as jp
import matplotlib.pyplot as plt
import mediapy as media
import mujoco
import numpy as np
from brax.training.agents.ppo import networks as ppo_networks
from brax.training.agents.ppo import train as ppo
from IPython.display import clear_output, display
from mujoco_playground import registry, wrapper
from mujoco_playground.config import locomotion_params

subprocess.run(["nvidia-smi"], check=True)
print("JAX:", metadata.version("jax"))
print("Flax:", metadata.version("flax"))
print("Brax:", metadata.version("brax"))
print("MuJoCo:", metadata.version("mujoco"))
print("Playground:", metadata.version("playground"))
print("JAX devices:", jax.devices())

## Environment and default PPO configuration

One environment contains one humanoid and one physics simulation. PPO creates
many copies of this environment on the GPU. Every copy uses the same policy.

The command has three values: forward speed, sideways speed, and turning speed.
The actor receives robot observations and this command. The critic also receives
privileged simulator information during training.

In [ ]:
#@title Load the Berkeley Humanoid task
ENV_NAME = "BerkeleyHumanoidJoystickFlatTerrain"
env_cfg = registry.get_default_config(ENV_NAME)
env = registry.load(ENV_NAME, config=env_cfg)
ppo_params = locomotion_params.brax_ppo_config(ENV_NAME)

print("Environment:", ENV_NAME)
print("Physics backend:", env_cfg.impl)
print("Episode length:", env_cfg.episode_length)
print("Default parallel environments:", ppo_params.num_envs)
print("Default environment steps:", ppo_params.num_timesteps)
print("Unroll length:", ppo_params.unroll_length)
print("Minibatches:", ppo_params.num_minibatches)
print("Learning rate:", ppo_params.learning_rate)
print("Actor layers:", ppo_params.network_factory.policy_hidden_layer_sizes)
print("Critic layers:", ppo_params.network_factory.value_hidden_layer_sizes)

## Smoke mode versus full training

Smoke mode uses fewer simulator copies and fewer environment steps. Its purpose
is to find installation, memory, saving, and rendering errors quickly.

Full mode keeps the repository's tuned settings: 8,192 parallel environments
and 150 million environment steps. Use full mode only after smoke mode passes.

In [ ]:
#@title Select training size and artifact location
if not RUN_FULL_TRAINING:
    ppo_params.num_timesteps = 262_144
    ppo_params.num_evals = 2
    ppo_params.num_envs = 512
    ppo_params.batch_size = 64
    ppo_params.num_minibatches = 8
    ppo_params.num_resets_per_eval = 1

SOURCE_REPOSITORY = "spinachfarmer/MuJoCo-Playground"
SOURCE_BRANCH = "codex/clean-berkeley-humanoid-notebook"
commit_api = (
    f"https://api.github.com/repos/{SOURCE_REPOSITORY}/commits/{SOURCE_BRANCH}"
)
try:
    with urllib.request.urlopen(commit_api, timeout=10) as response:
        SOURCE_COMMIT = json.load(response)["sha"]
except Exception as error:
    SOURCE_COMMIT = "unknown"
    print("Warning: could not record the Git commit:", error)

run_mode = "full" if RUN_FULL_TRAINING else "smoke"
run_time = datetime.now(timezone.utc).strftime("%Y-%m-%d_%H%M%S_UTC")
run_id = f"{run_time}_{run_mode}_seed{SEED}_{SOURCE_COMMIT[:7]}"

if SAVE_ARTIFACTS_TO_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    ARTIFACT_DIR = Path(
        "/content/drive/MyDrive/MuJoCo_Playground/BerkeleyHumanoid/runs"
    ) / run_id
else:
    ARTIFACT_DIR = Path("/content/mujoco_artifacts") / run_id

ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
print("Training mode:", "FULL" if RUN_FULL_TRAINING else "SMOKE")
print("Parallel environments:", ppo_params.num_envs)
print("Environment steps:", ppo_params.num_timesteps)
print("Source commit:", SOURCE_COMMIT)
print("Run ID:", run_id)
print("Artifacts:", ARTIFACT_DIR)

In [ ]:
#@title Define progress reporting before training
x_data = []
y_data = []
y_error = []


def progress(num_steps, metrics):
    x_data.append(int(num_steps))
    y_data.append(float(metrics["eval/episode_reward"]))
    y_error.append(float(metrics["eval/episode_reward_std"]))

    clear_output(wait=True)
    plt.figure(figsize=(7, 4))
    plt.errorbar(x_data, y_data, yerr=y_error, marker="o")
    plt.xlabel("Environment steps")
    plt.ylabel("Evaluation reward per episode")
    plt.title(f"Latest reward: {y_data[-1]:.2f}")
    plt.grid(alpha=0.25)
    plt.tight_layout()
    plt.savefig(ARTIFACT_DIR / "learning_curve_live.png", dpi=160)
    with open(ARTIFACT_DIR / "progress.json", "w") as file:
        json.dump({
            "steps": x_data,
            "episode_reward": y_data,
            "episode_reward_std": y_error,
        }, file, indent=2)
    display(plt.gcf())
    plt.close()


randomizer = registry.get_domain_randomizer(ENV_NAME)
CHECKPOINT_DIR = ARTIFACT_DIR / "checkpoints"
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
training_params = dict(ppo_params)
training_params["save_checkpoint_path"] = str(CHECKPOINT_DIR)
network_options = training_params.pop("network_factory")
network_factory = functools.partial(
    ppo_networks.make_ppo_networks,
    **network_options,
)

train_fn = functools.partial(
    ppo.train,
    **training_params,
    network_factory=network_factory,
    randomization_fn=randomizer,
    progress_fn=progress,
)

In [ ]:
#@title Train the shared PPO actor and critic
make_inference_fn, params, metrics = train_fn(
    environment=env,
    eval_env=registry.load(ENV_NAME, config=env_cfg),
    wrap_env_fn=wrapper.wrap_for_brax_training,
    seed=SEED,
)
print("Training finished.")

In [ ]:
#@title Save policy parameters, metrics, and the learning curve
params_cpu = jax.device_get(params)

with open(ARTIFACT_DIR / "policy_params.pkl", "wb") as file:
    pickle.dump(params_cpu, file)

run_summary = {
    "run_id": run_id,
    "run_started_utc": run_time,
    "source_repository": SOURCE_REPOSITORY,
    "source_branch": SOURCE_BRANCH,
    "source_commit": SOURCE_COMMIT,
    "environment": ENV_NAME,
    "full_training": RUN_FULL_TRAINING,
    "seed": SEED,
    "num_envs": int(ppo_params.num_envs),
    "num_timesteps": int(ppo_params.num_timesteps),
    "steps": x_data,
    "episode_reward": y_data,
    "episode_reward_std": y_error,
    "versions": {
        name: metadata.version(name)
        for name in ["playground", "mujoco", "brax", "jax", "flax"]
    },
}
with open(ARTIFACT_DIR / "run_summary.json", "w") as file:
    json.dump(run_summary, file, indent=2)

if x_data:
    plt.figure(figsize=(8, 4.5))
    plt.errorbar(x_data, y_data, yerr=y_error, marker="o")
    plt.xlabel("Environment steps")
    plt.ylabel("Evaluation reward per episode")
    plt.title("Berkeley Humanoid PPO learning curve")
    plt.grid(alpha=0.25)
    plt.tight_layout()
    plt.savefig(ARTIFACT_DIR / "learning_curve.png", dpi=180)
    plt.show()

print("Saved training artifacts to", ARTIFACT_DIR)

## Keyboard-style command rollout

Colab cannot provide a reliable real-time desktop keyboard loop. This cell uses
the same key meanings, but it reads a planned sequence from `KEY_SEQUENCE`.

- `W` and `S`: forward and backward
- `A` and `D`: move left and right
- `Q` and `E`: turn left and right
- `X`: stop

The policy does not receive joint instructions from these keys. It receives a
three-value velocity command and decides the 12 joint actions itself.

In [ ]:
#@title Free training memory and render the command sequence
del env, metrics, train_fn
gc.collect()
jax.clear_caches()

# The JAX physics implementation uses less GPU memory for this single rollout.
eval_cfg = registry.get_default_config(ENV_NAME)
eval_cfg.impl = "jax"
eval_env = registry.load(ENV_NAME, config=eval_cfg)

jit_reset = jax.jit(eval_env.reset)
jit_step = jax.jit(eval_env.step)
jit_policy = jax.jit(make_inference_fn(params_cpu, deterministic=True))

key_to_command = {
    "w": jp.array([1.0, 0.0, 0.0]),
    "s": jp.array([-0.5, 0.0, 0.0]),
    "a": jp.array([0.0, 0.5, 0.0]),
    "d": jp.array([0.0, -0.5, 0.0]),
    "q": jp.array([0.0, 0.0, 1.0]),
    "e": jp.array([0.0, 0.0, -1.0]),
    "x": jp.array([0.0, 0.0, 0.0]),
}
unknown_keys = sorted(set(KEY_SEQUENCE.lower()) - set(key_to_command))
if unknown_keys:
    raise ValueError(f"Unsupported keys: {unknown_keys}")

rng = jax.random.PRNGKey(SEED + 1)
state = jit_reset(rng)
state.info["phase_dt"] = 2 * jp.pi * eval_env.dt * 1.5
state.info["phase"] = jp.array([0, jp.pi])
rollout = []
steps_per_key = max(1, round(SECONDS_PER_KEY / float(eval_env.dt)))

for key in KEY_SEQUENCE.lower():
    command = key_to_command[key]
    for _ in range(steps_per_key):
        state.info["command"] = command
        action_rng, rng = jax.random.split(rng)
        action, _ = jit_policy(state.obs, action_rng)
        state = jit_step(state, action)
        rollout.append(jax.device_get(state))
        if bool(jax.device_get(state.done)):
            break
    if bool(jax.device_get(state.done)):
        print("The episode ended early because the humanoid fell.")
        break

render_every = 2
fps = 1.0 / float(eval_env.dt) / render_every
frames = eval_env.render(
    rollout[::render_every],
    camera="track",
    width=640,
    height=480,
)
video_path = ARTIFACT_DIR / "keyboard_command_rollout.mp4"
media.write_video(str(video_path), frames, fps=fps)
media.show_video(frames, fps=fps, loop=False)
print("Saved rollout video to", video_path)

## What is persistent?

- **GitHub:** notebook code and explanations after a commit and push.
- **Google Drive:** one timestamped folder per run when
  `SAVE_ARTIFACTS_TO_DRIVE` is `True`. Each folder contains the policy,
  metrics, learning curve, video, and source Git commit.
- **Temporary Colab storage:** artifacts under `/content` when Drive saving is
  disabled. Download them before the runtime disconnects.

Do not commit large policy checkpoints to the Git repository.